## **PPG訊號量測實作單元(一)：照過來照過來！各位客倌快來看看你的脈搏！**

> 在這次的實驗中，鐵克將會先沿用ECG心電訊號章節曾經教大家的內容來練習波峰偵測！

> 試著用PPG訊號求出自己心率的相關資料吧！

# 0. 先收錄一些會用到的公式和函式吧！

In [1]:
#@title
from google.colab import files
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

def find_peak_weird_detection(data, height, distance):
  peak_list = find_peaks(data, height=height, distance=distance)[0]
  if(peak_list.size != 0):
    print('總共找到', peak_list.size, '個Peak')
    peak_peak_list = (np.diff(peak_list))
    pp_mean = np.mean(peak_peak_list)
    counter = 0
    counter2 = 0
    counter3 = 0

    for i in peak_peak_list:
      if(i > pp_mean+100):
        counter2 = counter2 + 1
    
    print('發現有',counter2,'個「怪怪Peak」')

    weird_peak_list_start = np.zeros(counter2,int)
    weird_peak_list_end = np.zeros(counter2,int)
    weird_pp_list = np.zeros(counter2,int)
  

    for i in peak_peak_list:
      if(i > pp_mean+100):
        weird_peak_list_start[counter3] = peak_list[counter]
        weird_peak_list_end[counter3] = peak_list[counter+1]
        weird_pp_list[counter3] = i
        counter3 += 1
      counter = counter + 1

    pp_mean_fixed_1 = (pp_mean*(peak_peak_list.size))
    pp_mean_fixed_2 = pp_mean_fixed_1 - np.sum(weird_pp_list)
    pp_mean_fixed = int(pp_mean_fixed_2/(peak_peak_list.size - weird_pp_list.size))

  else:
    print('沒有找到任何Peak，請重新調整height和distance來尋找peak。')

  return peak_list, peak_peak_list, weird_peak_list_start, weird_peak_list_end, pp_mean_fixed 

# 1.將範例PPG訊號丟進程式庫吧！

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

# 2.看看真實的PPG訊號長什麼樣子吧！

In [ ]:
#來讀取剛上傳的資料吧！
df = pd.read_csv('範例PPG訊號.txt') #記得確認('xxxx')裡面的檔名唷！
data = np.array(df)
data2 = data[0 : , 0]

#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2,
    name='Original Plot'
))
fig.update_layout(
    title="測得的PPG訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出PPG訊號吧！
fig.show() 

# 3.沿用ECG訊號的波峰偵測程式算出心跳速率吧！

> 執行後若有發現怪怪Peak，試著調整height和distance吧！

> 觀察上面的PPG訊號，height調整可以參考波峰高度平均落在Y軸的多少呢？

> distance調整可以參考波峰之間間距從X軸上來看大概是多少呢？

In [ ]:
height = 90
distance = 600
(peak_list, pp_list, weird_p_start, weird_p_end, pp_mean) = find_peak_weird_detection(data2, height, distance)

In [ ]:
#再次檢驗看看檢測到的波形吧！
#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2,
    mode='lines',
    name='PPG訊號'
))

#把偵測到的波峰也標記在畫布上吧！
fig.add_trace(go.Scatter(
    x=peak_list,
    y=[data2[j] for j in peak_list],
    mode='markers',
    marker=dict(
        size=8,
        color='red',
        symbol='cross'
    ),
    name='分析之Peak位置'
))
fig.update_layout(
    title="測得的PPG訊號與波峰偵測結果",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出你的成果吧！
fig.show()

# 4.可以計算心跳速率了！

> 執行程式

> 觀察段落程式的說明

> 往下查看結果

In [ ]:
print('每個PPG訊號波峰是在資料位置',peak_list)

In [ ]:
peak_peak_list = (np.diff(peak_list)) #兩兩波峰相減計算波峰之間的距離
print('PPG訊號波峰之間分別距離幾個資料單位',peak_peak_list)

In [ ]:
peak_peak_list2 = peak_peak_list*0.001 #每個資料單位之間的距離是0.001秒
print('每個心跳時間是',peak_peak_list2,'秒')

In [ ]:
peak_peak_time = np.mean(peak_peak_list2) #把每個心跳時間加總取平均吧
print('每個心跳時間的平均為',peak_peak_time,'秒')

In [ ]:
heart_rate = (60/peak_peak_time) #60秒內心跳可以跳幾下呢
print('心跳速率為',heart_rate, '下/分鐘')

# 單元小結論：
---
#1.   PPG訊號會有明顯的波峰
#2.   偵測波峰是一件重要的是
#3.訊號波峰間距時間能夠推算心跳速率
